# 🌾 Harvestline — Machine Learning Crop Yield Prediction Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This Google Colab notebook provides an end-to-end Machine Learning workflow for predicting global crop yields based on FAO & World Bank climate records (1990–2013).

### Notebook Workflow:
1. **Environment Setup & Data Loading**
2. **Exploratory Data Analysis (EDA)**
3. **Domain Feature Engineering** (`rain_temp_interaction`, `pesticide_per_rainfall`, `years_since_start`)
4. **Data Preprocessing & Pipeline Assembly** (`ColumnTransformer` + One-Hot Encoding)
5. **Model Benchmarking & Selection** (Linear Regression vs. Decision Tree vs. Random Forest)
6. **Model Evaluation** ($R^2$, RMSE, MAE & Residual Analysis)
7. **Feature Importance Extraction**
8. **Pipeline Serialization & Lookup Export** (`joblib.dump`)


In [ ]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print("Libraries loaded successfully.")


## 2. Data Loading & Initial Exploration

In [ ]:
# Load dataset
data_url = '../data/yield_df.csv'
if not os.path.exists(data_url):
    data_url = 'yield_df.csv'

df = pd.read_csv(data_url)
print(f"Dataset Shape: {df.shape}")
df.head()


In [ ]:
# Check column types and summary statistics
print(df.info())
df.describe()


## 3. Feature Engineering

We engineer three domain-specific interaction features:
- `rain_temp_interaction`: $\text{Rainfall (mm)} \times \text{Temperature (°C)}$
- `pesticide_per_rainfall`: $\frac{\text{Pesticides (tonnes)}}{\text{Rainfall (mm)} + 1e-5}$
- `years_since_start`: $\text{Year} - 1990$


In [ ]:
START_YEAR = 1990

df['rain_temp_interaction'] = df['average_rain_fall_mm_per_year'] * df['avg_temp']
df['pesticide_per_rainfall'] = df['pesticides_tonnes'] / (df['average_rain_fall_mm_per_year'] + 1e-5)
df['years_since_start'] = df['Year'] - START_YEAR

print("Engineered features added successfully.")
df[['Year', 'years_since_start', 'rain_temp_interaction', 'pesticide_per_rainfall']].head()


## 4. Pipeline Assembly & Preprocessing

We construct a `ColumnTransformer` that handles Categorical One-Hot Encoding for `Area` and `Item`, while passing numerical features through directly.

In [ ]:
num_features = [
    "Year",
    "average_rain_fall_mm_per_year",
    "pesticides_tonnes",
    "avg_temp",
    "rain_temp_interaction",
    "pesticide_per_rainfall",
    "years_since_start"
]
cat_features = ["Area", "Item"]
target = "hg/ha_yield"

X = df[num_features + cat_features]
y = df[target]

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ]
)


## 5. Model Training & Evaluation Benchmark

We train and benchmark three models:
1. **Linear Regression** (Linear baseline)
2. **Single Decision Tree Regressor** (Non-linear baseline)
3. **Random Forest Regressor** (Ensemble pipeline)


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
}

results = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    
    results[name] = {"R2": r2, "RMSE": rmse, "MAE": mae, "pipeline": pipe}
    print(f"=== {name} ===\n  R2 Score: {r2:.4f}\n  RMSE: {rmse:.2f} hg/ha ({rmse/10000:.2f} t/ha)\n  MAE: {mae:.2f} hg/ha\n")


## 6. Serializing Final Pipeline & Saving Model Artifacts

In [ ]:
final_pipeline = results["Random Forest"]["pipeline"]
output_model_path = "../backend/model/crop_yield_pipeline.pkl"
if not os.path.exists(os.path.dirname(output_model_path)):
    output_model_path = "crop_yield_pipeline.pkl"

joblib.dump(final_pipeline, output_model_path)
print(f"Final Random Forest pipeline successfully exported to {output_model_path}")
